# 05. 타임아웃 재시도 폴백

일시적인 실패를 몇 번까지 다시 시도하고 언제 대체 경로를 선택해야 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 Runnable 재시도·폴백 실습의 고정 패키지 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 OpenAI 키 값은 출력하지 않습니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# timeout은 초 단위 대기 상한, max_retries는 SDK의 추가 시도 상한으로 설정합니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## 실제 호출의 제한 설정

모델 객체에 타임아웃과 최대 재시도 횟수가 설정되었음을 직렬화 가능한 설정에서 확인합니다.

In [4]:
# 실제 API가 정상인 현재 조건의 응답을 로컬 오류 시뮬레이션과 분리합니다.
print("기본 모델 타입:", type(model).__name__)
healthy = model.invoke("재시도는 왜 횟수를 제한해야 하나요? 한 문장으로 답하세요.")
print(healthy.text)

기본 모델 타입: ChatOpenAI


무한 재시도는 장애를 더 악화시키고 자원을 소진시킬 수 있으므로, 피해를 제한하고 빠르게 복구·전환하기 위해 횟수를 제한합니다.


## 결정적인 로컬 시뮬레이션

네트워크 장애를 꾸며 내지 않고, 실패하는 Runnable로 제어 흐름만 검증합니다.

In [5]:
# primary는 항상 TimeoutError를 내고 fallback은 대체 문자열을 반환합니다.
from langchain_core.runnables import RunnableLambda

attempts = {"primary": 0}
def fail_primary(value: str) -> str:
    attempts["primary"] += 1
    raise TimeoutError("수업용 로컬 타임아웃")

primary = RunnableLambda(fail_primary)
fallback = RunnableLambda(lambda value: f"대체 경로: {value}")

## 제한된 재시도

첫 시도만 실패하고 두 번째 시도에 성공하는 함수로 실제 재시도 횟수를 셉니다.

In [6]:
# 첫 실행만 실패하는 함수에 with_retry의 총 시도 상한 2회를 적용합니다.
retry_attempts = {"count": 0}
def transient(value: str) -> str:
    retry_attempts["count"] += 1
    if retry_attempts["count"] == 1:
        raise TimeoutError("수업용 일시 오류")
    return f"재시도 성공: {value}"

retrying = RunnableLambda(transient).with_retry(
    retry_if_exception_type=(TimeoutError,), stop_after_attempt=2
)
print(retrying.invoke("질문"))
print("총 시도 횟수:", retry_attempts["count"])

재시도 성공: 질문
총 시도 횟수: 2


## 실패 재현

In [7]:
# 폴백 없이 primary 한 번의 예상한 TimeoutError와 시도 횟수를 확인합니다.
try:
    primary.invoke("질문")
except TimeoutError as error:
    print("로컬 시뮬레이션:", type(error).__name__)
    print("시도 횟수:", attempts["primary"])
else:
    raise AssertionError("기본 경로는 실패해야 합니다.")

로컬 시뮬레이션: TimeoutError
시도 횟수: 1


## 폴백 연결

In [8]:
# TimeoutError에만 with_fallbacks를 적용해 대체 경로 반환값을 확인합니다.
safe_chain = primary.with_fallbacks([fallback], exceptions_to_handle=(TimeoutError,))
result = safe_chain.invoke("질문")
print(result)
print("누적 기본 경로 시도:", attempts["primary"])
assert result.startswith("대체 경로")

대체 경로: 질문
누적 기본 경로 시도: 2
